# 04. Production exports, dashboard and updates
Start with notebook 01. Use the same Drive project folder. Run one stage cell at a time and share its output for review. Never use Run all across stages.


## Connect Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Get the code: clone once, pull on later runs
The repository stays on Drive so saved artifacts survive runtime disconnects. An existing nonempty folder without Git needs review before cloning. Stop if Git reports an error; existing conflicts need resolution before continuing. No files are reset.


In [ ]:
%%bash
set -e
REPO_DIR='/content/drive/MyDrive/FoodPriceWatch'
REPO_URL='https://github.com/mrchowdhury587/foodprice_watch.git'
if [ ! -d "$REPO_DIR/.git" ]; then
    git clone -b main "$REPO_URL" "$REPO_DIR"
else
    git -C "$REPO_DIR" pull --no-rebase origin main
fi


## Initialize project runtime
Persistent artifacts are saved under the Drive project folder.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, csv, runpy
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
assert (ROOT / 'src/fpw/runner.py').is_file(), 'Repository is missing on Drive'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('FPW_THREADS', str(os.cpu_count() or 2))
def run_python(path, *arguments, module=False):
    previous = sys.argv[:]
    try:
        sys.argv = [path, *arguments]
        if module:
            runpy.run_module(path, run_name='__main__')
        else:
            runpy.run_path(path, run_name='__main__')
    except SystemExit as result:
        if result.code not in (None, 0):
            raise RuntimeError(f'{path} failed: {result.code}') from result
    finally:
        sys.argv = previous



## 3. Install runtime packages


In [ ]:
%%bash
set -e
cd /content/drive/MyDrive/FoodPriceWatch
python -m pip install -q -r requirements-colab.txt


## 4. Startup verification
A failed check stops execution. Missing or changed finished outputs are reset to pending by the project checker; paste failures back before continuing.


In [ ]:
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        assert tmp.resolve().is_relative_to(ROOT.resolve())
        tmp.unlink()
        print('Removed interrupted-write temporary file:', tmp.relative_to(ROOT))
run_python('fpw.check', module=True)
def run_unit(sid):
    from fpw.runner import run
    assert run(sid) == 0, 'STOP: stage failed'
    ledger = json.loads((ROOT / 'state/stages.json').read_text())
    stage = next(s for s in ledger['stages'] if s['id'] == sid)
    assert stage['status'] == 'done', 'STOP: stage was not finalized'
    summary = []
    for name, expected in stage['output_sha256'].items():
        path = ROOT / name
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        assert actual == expected, 'STOP: output hash mismatch: ' + name
        rows = None
        if path.suffix == '.csv':
            with path.open(newline='') as f:
                rows = sum(1 for _ in csv.DictReader(f))
        elif path.suffix == '.parquet':
            import pyarrow.parquet as pq
            rows = pq.read_metadata(path).num_rows
        summary.append(dict(file=name, rows=rows, sha256=actual))
    assert summary, 'STOP: no finalized output hashes'
    print(json.dumps(dict(unit=sid, outputs=summary, check='PASS', git_commit=stage.get('git_commit')), indent=2))


## S30: Production training and exports
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


## S31: Local dashboard instructions
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


## S32: Production update workflow
**Not implemented yet.** No execution command is provided. Prerequisites and required decisions must be verified before this stage is implemented.


Testing requires committed pre-registration and existing checkpoint enforcement. Do not rerun earlier sealed tests. Keep models, exports and data out of GitHub.
